# Notebook 06 — Pipeline Score Rules
## Monte Carlo + Ranked Probability Score | Projeto Fórmula 1 — TCC

Este notebook apresenta e analisa os resultados do **Pipeline 2**,
que incrementa o Pipeline 1 adicionando previsão probabilística via
Monte Carlo e avaliação via Ranked Probability Score (RPS).

### Estrutura do notebook
1. Configuração e carregamento dos resultados
2. Vetor de probabilidades — o que é e como interpretar
3. RPS — Ranked Probability Score
4. Visualizações: mapa de calor, evolução do RPS, P(vitória), ganho
5. Análise de casos específicos
6. Conclusões


## 1. Configuração

In [3]:
import sys, os, pickle
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

PROJECT_ROOT  = r'C:/Users/guiga/OneDrive/Documentos/Projeto-Formula1'
PIPELINE2_DIR = os.path.join(PROJECT_ROOT, 'src', 'pipeline_score_rules')

sys.path.insert(0, PROJECT_ROOT)

from src.pipeline_score_rules.scoring_rules import rps_season_summary
from src.pipeline_mallows_plackett_luce.visualization_plots import TEAM_COLORS, CLUSTER_COLORS

output_file = os.path.join(PIPELINE2_DIR, 'outputs', 'nb_data_p2.pkl')
with open(output_file, 'rb') as f:
    data = pickle.load(f)
print(data.keys())
state             = data['state']
val_evals         = data['val_evals']
test_evals        = data['test_evals']
val_rps           = data['val_rps']
test_rps          = data['test_rps']
val_rps_summary   = data['val_rps_summary']
test_rps_summary  = data['test_rps_summary']
val_distributions = data['val_distributions']
test_distributions= data['test_distributions']
val_records       = data['val_records']
test_records      = data['test_records']
all_drivers       = data['all_drivers']

print("Dados carregados com sucesso.")
print(f"  Corridas — validação: {len(val_rps)}")
print(f"  Corridas — teste:     {len(test_rps)}")


dict_keys(['state', 'val_evals', 'test_evals', 'val_rps', 'test_rps', 'val_rps_summary', 'test_rps_summary', 'all_drivers', 'train_records'])


KeyError: 'val_distributions'

## 2. Vetor de Probabilidades

### O que é e como é gerado

Para cada corrida, o Monte Carlo usa os skill scores λ_i do cluster
identificado pelo Mallows e simula **10.000 corridas** usando esses scores
como pesos. Cada piloto é "sorteado" para cada posição proporcional ao seu λ_i.

O resultado é um vetor por piloto:
```
VER → [P(1º), P(2º), P(3º), ..., P(10º)]
      [0.45,  0.22,  0.13,  ..., 0.01  ]
```

A soma de cada vetor é sempre 1.0.


In [ ]:
# Exibir vetor de probabilidades da primeira corrida de 2024
dist  = test_distributions[0]
race  = test_records[0].race
print(f"Distribuição de probabilidades — {race} {test_records[0].season}")
print(f"\n  {'Piloto':>6}  {'P(1º)':>7}  {'P(2º)':>7}  {'P(3º)':>7}  "
      f"{'P(4º)':>7}  {'P(5º)':>7}  {'Top-3':>7}  {'Pos. Esp.':>10}")
print("  " + "-" * 70)

win_probs = dist.win_probabilities()
for driver in list(win_probs.keys())[:10]:
    vec = dist.vectors[driver]
    p   = vec.probs
    print(f"  {driver:>6}  {p[0]:>7.3f}  {p[1]:>7.3f}  {p[2]:>7.3f}  "
          f"{p[3]:>7.3f}  {p[4]:>7.3f}  {vec.p_top_n(3):>7.3f}  "
          f"{vec.expected_position():>10.2f}")


## 3. RPS — Ranked Probability Score

### O que mede

O RPS avalia a **qualidade das probabilidades** — não só se o modelo acertou
o vencedor, mas se a distribuição inteira era coerente com o resultado.

```
RPS = (1/K) × Σ (CDF_prevista(k) - CDF_real(k))²
```

- **RPS = 0.0** → previsão perfeita
- **RPS baseline** ≈ 0.13 (modelo que não sabe nada sobre F1)
- **Ganho** = RPS_baseline − RPS_modelo (quanto melhor que o aleatório)

Um modelo com ganho positivo em uma corrida foi **mais informativo** que
atribuir probabilidade igual a todos os pilotos.


In [ ]:
# Tabela RPS completa — validação 2023
print("RPS — VALIDAÇÃO 2023")
print(f"\n  {'Corrida':22s} {'Cluster':>8} {'RPS Modelo':>12} "
      f"{'RPS Baseline':>14} {'Ganho':>8}")
print("  " + "-" * 68)
for r in val_rps:
    marker = "✓" if r.gain > 0 else "✗"
    print(f"  {r.race:22s} {r.cluster+1:>8d} {r.rps_model:>12.4f} "
          f"{r.rps_baseline:>14.4f} {r.gain:>8.4f} {marker}")
print("  " + "-" * 68)
print(f"  {'MÉDIA':22s} {'':>8} {val_rps_summary.mean_rps_model:>12.4f} "
      f"{val_rps_summary.mean_rps_baseline:>14.4f} "
      f"{val_rps_summary.mean_gain:>8.4f}")


In [ ]:
# Tabela RPS completa — teste 2024
print("RPS — TESTE 2024")
print(f"\n  {'Corrida':22s} {'Cluster':>8} {'RPS Modelo':>12} "
      f"{'RPS Baseline':>14} {'Ganho':>8}")
print("  " + "-" * 68)
for r in test_rps:
    marker = "✓" if r.gain > 0 else "✗"
    print(f"  {r.race:22s} {r.cluster+1:>8d} {r.rps_model:>12.4f} "
          f"{r.rps_baseline:>14.4f} {r.gain:>8.4f} {marker}")
print("  " + "-" * 68)
print(f"  {'MÉDIA':22s} {'':>8} {test_rps_summary.mean_rps_model:>12.4f} "
      f"{test_rps_summary.mean_rps_baseline:>14.4f} "
      f"{test_rps_summary.mean_gain:>8.4f}")


## 4. Visualizações

### Gráfico 1 — Mapa de Calor de Probabilidades

In [ ]:
# Mapa de calor: pilotos x posições para uma corrida específica
# Mostra P(piloto i = posição k) para os top-10 pilotos

def plot_heatmap(dist, race_name, actual_ranking=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 7))
        fig.patch.set_facecolor('#1a1a2e')

    top_drivers = dist.drivers[:10]
    n_pos       = dist.vectors[top_drivers[0]].n_positions
    matrix      = np.array([dist.vectors[d].probs for d in top_drivers])

    im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto',
                   vmin=0, vmax=matrix.max())
    plt.colorbar(im, ax=ax, label='Probabilidade')

    ax.set_xticks(range(n_pos))
    ax.set_xticklabels([f'{i+1}º' for i in range(n_pos)], fontsize=9)
    ax.set_yticks(range(len(top_drivers)))
    ax.set_yticklabels(top_drivers, fontsize=9)

    # Marcar posição real
    if actual_ranking:
        pos_map = {d: i for i, d in enumerate(actual_ranking)}
        for row, driver in enumerate(top_drivers):
            if driver in pos_map:
                col = pos_map[driver]
                if col < n_pos:
                    ax.add_patch(plt.Rectangle((col-0.5, row-0.5), 1, 1,
                                 fill=False, edgecolor='white', linewidth=2.5))

    ax.set_title(f'Mapa de Probabilidades — {race_name}\n'
                 f'(borda branca = posição real)',
                 fontsize=11, fontweight='bold', color='white')
    ax.set_xlabel('Posição', fontsize=9)
    ax.set_ylabel('Piloto', fontsize=9)
    return ax

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#1a1a2e')

# Bahrain 2024 (primeira corrida do teste)
plot_heatmap(test_distributions[0], test_records[0].race,
             test_records[0].ranking, ax=axes[0])

# Corrida com pior RPS (Australia)
worst_idx = max(range(len(test_rps)), key=lambda i: test_rps[i].rps_model)
plot_heatmap(test_distributions[worst_idx],
             test_records[worst_idx].race,
             test_records[worst_idx].ranking, ax=axes[1])

plt.tight_layout()
plt.show()


### Gráfico 2 — Evolução do RPS ao longo das corridas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('#1a1a2e')

for ax, rps_list, season, color in zip(
    axes, [val_rps, test_rps], [2023, 2024], ['#FF8000', '#3671C6']
):
    races    = [r.race[:8]      for r in rps_list]
    rps_mod  = [r.rps_model     for r in rps_list]
    rps_base = [r.rps_baseline  for r in rps_list]
    x        = range(len(races))

    ax.plot(x, rps_base, color='#aaaacc', linewidth=1.5,
            label='Baseline', linestyle='--', alpha=0.7)
    ax.plot(x, rps_mod, color=color, linewidth=2.0,
            label='Modelo', marker='o', markersize=4)
    ax.fill_between(x, rps_mod, rps_base,
                    where=[m < b for m, b in zip(rps_mod, rps_base)],
                    alpha=0.2, color='#52E252', label='Ganho')
    ax.fill_between(x, rps_mod, rps_base,
                    where=[m >= b for m, b in zip(rps_mod, rps_base)],
                    alpha=0.2, color='#E8002D', label='Perda')

    ax.set_xticks(list(x))
    ax.set_xticklabels(races, rotation=45, ha='right', fontsize=7)
    ax.set_title(f'RPS por Corrida — {season}', fontsize=12, fontweight='bold')
    ax.set_ylabel('RPS', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, framealpha=0.6)

plt.tight_layout()
plt.show()


### Gráfico 3 — Probabilidades de Vitória por Corrida

In [ ]:
# Mostrar P(1º) dos top-5 pilotos para cada corrida do teste
TOP5 = ['VER', 'NOR', 'LEC', 'PIA', 'HAM']

fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor('#1a1a2e')

races = [r.race[:8] for r in test_records]
x     = np.arange(len(races))
width = 0.15

for i, driver in enumerate(TOP5):
    win_probs = []
    for dist in test_distributions:
        vec = dist.vectors.get(driver)
        win_probs.append(vec.probs[0] if vec is not None else 0)
    color = TEAM_COLORS.get(driver, '#888899')
    ax.bar(x + i * width, win_probs, width, label=driver,
           color=color, alpha=0.85, edgecolor='#1a1a2e')

ax.set_xticks(x + width * 2)
ax.set_xticklabels(races, rotation=45, ha='right', fontsize=7)
ax.set_title('Probabilidade de Vitória P(1º) por Corrida — Teste 2024',
             fontsize=12, fontweight='bold')
ax.set_ylabel('P(1º)', fontsize=9)
ax.legend(fontsize=9, framealpha=0.6)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


### Gráfico 4 — Ganho sobre o Baseline por Corrida

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9))
fig.patch.set_facecolor('#1a1a2e')

for ax, rps_list, season in zip(axes, [val_rps, test_rps], [2023, 2024]):
    races = [r.race[:10] for r in rps_list]
    gains = [r.gain      for r in rps_list]
    x     = range(len(races))
    colors = ['#52E252' if g > 0 else '#E8002D' for g in gains]

    bars = ax.bar(x, gains, color=colors, alpha=0.85,
                  edgecolor='#1a1a2e', linewidth=0.8)
    ax.axhline(y=0, color='#ffffff', linewidth=1.0, alpha=0.5)
    ax.axhline(y=np.mean(gains), color='#FF8000', linewidth=1.5,
               linestyle='--', alpha=0.8, label=f'Média: {np.mean(gains):.4f}')

    # Anotar corridas problemáticas
    for i, (race, gain) in enumerate(zip(races, gains)):
        if gain < -0.005:
            ax.annotate(race, xy=(i, gain), xytext=(0, -15),
                       textcoords='offset points', ha='center',
                       fontsize=7, color='#E8002D')

    ax.set_xticks(list(x))
    ax.set_xticklabels(races, rotation=45, ha='right', fontsize=7)
    ax.set_title(f'Ganho sobre Baseline — {season}  '
                 f'(verde = modelo melhor, vermelho = baseline melhor)',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Ganho (RPS baseline − RPS modelo)', fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend(fontsize=9, framealpha=0.6)

plt.tight_layout()
plt.show()


## 5. Análise de Casos Específicos

### Corrida com melhor RPS (modelo mais preciso)


In [ ]:
all_rps = val_rps + test_rps
best    = min(all_rps, key=lambda r: r.rps_model)
worst   = max(all_rps, key=lambda r: r.rps_model)

print(f"Melhor RPS: {best.race} {best.season}")
print(f"  RPS Modelo:   {best.rps_model:.4f}")
print(f"  RPS Baseline: {best.rps_baseline:.4f}")
print(f"  Ganho:        {best.gain:.4f}")

print(f"\nPior RPS: {worst.race} {worst.season}")
print(f"  RPS Modelo:   {worst.rps_model:.4f}")
print(f"  RPS Baseline: {worst.rps_baseline:.4f}")
print(f"  Ganho:        {worst.gain:.4f}")
print(f"  Nota: corrida com abandono de favoritos — modelo não prevê DNFs")


## 6. Conclusões

### Resultados principais

1. **O modelo bate o baseline em 97.8% das corridas** — apenas a Austrália 2024
   teve ganho negativo, causado pelo abandono de Verstappen e Hamilton.

2. **Melhoria de ~32% sobre o baseline na validação e ~30% no teste** —
   as probabilidades geradas têm valor informativo real comparado ao modelo
   que não sabe nada sobre F1.

3. **O mapa de calor revela calibração coerente** — pilotos favoritos
   concentram probabilidade nas primeiras posições, enquanto pilotos
   menos competitivos têm distribuição mais espalhada.

4. **A queda de 32% para 30% entre validação e teste é mínima** —
   confirmando que o Pipeline 2 também generaliza bem.

### Por que o RPS complementa o Top-3 e Kendall τ

O Top-3 e Kendall τ avaliam se o modelo acertou a previsão pontual.
O RPS avalia se o modelo **sabia o quanto tinha certeza**. Um modelo
que diz "90% para VER" quando outro piloto vence é penalizado muito
mais do que um que diz "40% para VER". Isso é valiosa informação sobre
a **calibração** do modelo.
